# Build Dataset

Two approaches to building `large_joined_sample.csv` from the Semantic Scholar bulk release.
Both produce the same output schema; pick one and run its section.

- **Approach A — Reservoir sampling** (Massimo's original): uniform random sample across the
  entire corpus. Reads every shard — slow but maximally unbiased.
- **Approach B — Single shard** (fast): loads one full paper shard into memory (~75 s, ~1 GB RAM),
  then streams one abstract shard matching against it and stops at TARGET. ~4 minutes total.

  **Why not just read the same shard index from both datasets?**
  `abstracts` has 30 shards and `papers` has 60 — different shard counts mean different hash
  boundaries. Shard 0 of `abstracts` maps across all 60 paper shards, giving only ~58 matches
  per paper shard instead of all 1500. Loading a full paper shard (~3.7 M papers) and scanning
  one abstract shard (~5 M papers) gives ~120 K overlapping corpusids — enough to hit
  TARGET=1500 after scanning only ~60 K abstract records.

Shared infrastructure (imports, API config, helpers, release resolution) is in the cells
below and must be run before either approach.

---
## Shared infrastructure

In [22]:
import json
import gzip
import random
import requests
import pandas as pd
from tqdm import tqdm

API_BASE = "https://api.semanticscholar.org/datasets/v1"
API_KEY  = "TsqxUdfGI04FzZWTOsQKXaQMn1JtCtzl4E2QneSu"
HEADERS  = {"x-api-key": API_KEY}

JOINED_OUTPUT_CSV     = "large_joined_sample.csv"
JOINED_OUTPUT_PARQUET = "large_joined_sample.parquet"

print("Config loaded")

Config loaded


In [23]:
def stream_gz_jsonl(url, headers, max_records=None):
    """Stream a gzipped JSONL file from a URL, yielding one parsed record at a time."""
    with requests.get(url, headers=headers, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        with gzip.GzipFile(fileobj=resp.raw) as gz:
            for i, line in enumerate(gz):
                if not line:
                    continue
                yield json.loads(line.decode("utf-8"))
                if max_records is not None and (i + 1) >= max_records:
                    break


def get_dataset_files(release_id, dataset_name):
    """Return the list of shard URLs for a dataset in a given release."""
    url  = f"{API_BASE}/release/{release_id}/dataset/{dataset_name}"
    resp = requests.get(url, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    return list(resp.json()["files"])


def normalize_and_save(joined_df, output_csv, output_parquet):
    """Normalize column types, select final columns, and write CSV + Parquet."""
    for col in ["title", "abstract", "venue"]:
        if col not in joined_df.columns:
            joined_df[col] = ""
        joined_df[col] = joined_df[col].fillna("").astype(str)

    for col in ["authors", "s2fieldsofstudy"]:
        if col not in joined_df.columns:
            joined_df[col] = [[] for _ in range(len(joined_df))]

    for col in ["citationcount", "referencecount", "year", "url"]:
        if col not in joined_df.columns:
            joined_df[col] = None

    joined_df = joined_df[
        ["corpusid", "title", "abstract", "year", "venue",
         "authors", "s2fieldsofstudy", "citationcount", "referencecount", "url"]
    ].reset_index(drop=True)

    joined_df.to_parquet(output_parquet)
    joined_df.to_csv(output_csv, index=False)

    print(f"Saved {output_csv} — shape: {joined_df.shape}")
    print(f"title coverage   : {(joined_df['title'].str.len() > 0).mean():.2%}")
    print(f"abstract coverage: {(joined_df['abstract'].str.len() > 0).mean():.2%}")
    print(joined_df.head(3))
    return joined_df

In [24]:
# Resolve latest release
resp = requests.get(f"{API_BASE}/release/latest", headers=HEADERS, timeout=60)
resp.raise_for_status()
latest_info = resp.json()

RELEASE_ID = (
    latest_info.get("release_id")
    or latest_info.get("releaseId")
    or latest_info.get("release")
    or latest_info.get("id")
)

print("Using release:", RELEASE_ID)

Using release: 2026-03-10


---
## Approach A — Reservoir sampling

Uniform random sample across the entire corpus. Reads all shards of both `abstracts` and
`papers` — use for a maximally unbiased corpus. Slow (minutes to hours depending on corpus size).

This is Massimo's original approach, refactored to use the shared helpers above.

In [ ]:
# ---------- Approach A config ----------
RANDOM_SEED            = 42
TARGET_ABSTRACT_SAMPLE = 3000   # change for larger runs
ABSTRACT_FILES_TO_READ = None   # None = all shards
PAPERS_FILES_TO_READ   = None   # None = all shards
ABSTRACT_CHECKPOINT_EVERY = 500
PAPER_CHECKPOINT_EVERY    = 250

ABSTRACT_SAMPLE_PATH = "large_abstract_sample.parquet"
PAPERS_METADATA_PATH = "large_papers_metadata.parquet"

rng = random.Random(RANDOM_SEED)

In [ ]:
def reservoir_add(sample, item, max_size, seen_count, rng):
    if len(sample) < max_size:
        sample.append(item)
    else:
        j = rng.randint(0, seen_count - 1)
        if j < max_size:
            sample[j] = item

In [ ]:
# ---------- step 1: reservoir-sample abstracts ----------
abstract_files = get_dataset_files(RELEASE_ID, "abstracts")
rng.shuffle(abstract_files)
if ABSTRACT_FILES_TO_READ is not None:
    abstract_files = abstract_files[:ABSTRACT_FILES_TO_READ]

sampled_abstract_rows = []
seen_abstract_records = 0

for file_idx, file_url in enumerate(abstract_files, 1):
    print(f"\n[abstracts] reading file {file_idx}/{len(abstract_files)}")

    for rec in tqdm(stream_gz_jsonl(file_url, HEADERS), desc=f"abstracts shard {file_idx}", leave=False):
        corpusid = str(rec.get("corpusid") or "").strip()
        abstract = str(rec.get("abstract") or "").strip()
        if not corpusid or not abstract:
            continue

        seen_abstract_records += 1
        reservoir_add(sampled_abstract_rows, {"corpusid": corpusid, "abstract": abstract},
                      TARGET_ABSTRACT_SAMPLE, seen_abstract_records, rng)

        if seen_abstract_records % ABSTRACT_CHECKPOINT_EVERY == 0:
            pd.DataFrame(sampled_abstract_rows).to_parquet(ABSTRACT_SAMPLE_PATH)
            print(f"  seen: {seen_abstract_records:,} | sample: {len(sampled_abstract_rows):,}")

abstracts_df = pd.DataFrame(sampled_abstract_rows)
abstracts_df["corpusid"] = abstracts_df["corpusid"].astype(str)
abstracts_df.to_parquet(ABSTRACT_SAMPLE_PATH)
print(f"\nAbstracts sampled: {abstracts_df.shape}")

target_corpusids = set(abstracts_df["corpusid"].tolist())

In [ ]:
# ---------- step 2: match metadata from papers ----------
paper_files = get_dataset_files(RELEASE_ID, "papers")
if PAPERS_FILES_TO_READ is not None:
    paper_files = paper_files[:PAPERS_FILES_TO_READ]

matched_rows = []
matched_ids  = set()

for file_idx, file_url in enumerate(paper_files, 1):
    print(f"\n[papers] reading file {file_idx}/{len(paper_files)}")

    for rec in tqdm(stream_gz_jsonl(file_url, HEADERS), desc=f"papers shard {file_idx}", leave=False):
        corpusid = str(rec.get("corpusid") or "").strip()
        if corpusid in target_corpusids and corpusid not in matched_ids:
            matched_rows.append({
                "corpusid":        corpusid,
                "title":           rec.get("title", ""),
                "year":            rec.get("year"),
                "venue":           rec.get("venue", ""),
                "authors":         rec.get("authors", []),
                "s2fieldsofstudy": rec.get("s2fieldsofstudy", []),
                "citationcount":   rec.get("citationcount", 0),
                "referencecount":  rec.get("referencecount", 0),
                "url":             rec.get("url", ""),
            })
            matched_ids.add(corpusid)

            if len(matched_ids) % PAPER_CHECKPOINT_EVERY == 0:
                ckpt = pd.DataFrame(matched_rows)
                ckpt.to_parquet(PAPERS_METADATA_PATH)
                abstracts_df.merge(ckpt, on="corpusid", how="inner").to_csv(JOINED_OUTPUT_CSV, index=False)
                print(f"  checkpoint | matched: {len(matched_ids):,} / {len(target_corpusids):,}")

        if len(matched_ids) == len(target_corpusids):
            break

    print(f"  done file {file_idx} | matched: {len(matched_ids):,} / {len(target_corpusids):,}")
    if len(matched_ids) == len(target_corpusids):
        print("Matched all.")
        break

papers_df = pd.DataFrame(matched_rows)
papers_df["corpusid"] = papers_df["corpusid"].astype(str)
papers_df.to_parquet(PAPERS_METADATA_PATH)

In [ ]:
# ---------- step 3: join and save ----------
joined_df = abstracts_df.merge(papers_df, on="corpusid", how="inner")
joined_df = normalize_and_save(joined_df, JOINED_OUTPUT_CSV, JOINED_OUTPUT_PARQUET)

---
## Approach B — Single shard (fast)

Inverts the usual order to avoid the multi-shard fallback:

1. **Load entire paper shard into memory** as a corpusid → metadata dict (~75 s, ~1 GB RAM).
   One paper shard covers 1/60 of the corpus (~3.7 M papers).
2. **Stream one abstract shard**, match each record against the dict on the fly, stop at TARGET.
   One abstract shard covers 1/30 of the corpus (~5 M papers). Expected overlap with one paper
   shard is ~120 K records, so 1500 matches are found after scanning ~60 K abstract records.

No fallback needed. Total: ~4 minutes on a normal PC.

In [25]:
# ---------- Approach B config ----------
SHARD_IDX = 0     # which shard to read from both datasets
TARGET    = 1500  # number of papers to collect

In [26]:
abstract_files = get_dataset_files(RELEASE_ID, "abstracts")
paper_files    = get_dataset_files(RELEASE_ID, "papers")

print(f"Abstract shards : {len(abstract_files)}")
print(f"Paper shards    : {len(paper_files)}")
print(f"Will load paper shard {SHARD_IDX}, then scan abstract shard {SHARD_IDX}")

Abstract shards : 30
Paper shards    : 60
Will load paper shard 0, then scan abstract shard 0


In [27]:
# ---------- step 1: load entire paper shard into memory ----------
# ~75 s, ~1 GB RAM for a shard of ~3.7 M papers.
# Stores corpusid -> metadata dict for fast O(1) lookup in step 2.
paper_lookup = {}

for rec in tqdm(stream_gz_jsonl(paper_files[SHARD_IDX], HEADERS), desc=f"loading paper shard {SHARD_IDX}"):
    corpusid = str(rec.get("corpusid") or "").strip()
    if corpusid:
        paper_lookup[corpusid] = {
            "title":           rec.get("title", ""),
            "year":            rec.get("year"),
            "venue":           rec.get("venue", ""),
            "authors":         rec.get("authors", []),
            "s2fieldsofstudy": rec.get("s2fieldsofstudy", []),
            "citationcount":   rec.get("citationcount", 0),
            "referencecount":  rec.get("referencecount", 0),
            "url":             rec.get("url", ""),
        }

print(f"Loaded {len(paper_lookup):,} papers from shard {SHARD_IDX} into memory")

loading paper shard 0: 4868915it [01:20, 60467.14it/s]

Loaded 4,868,915 papers from shard 0 into memory


In [28]:
# ---------- step 2: stream abstract shard, join on the fly, stop at TARGET ----------
# Scans the abstract shard record by record, checking each corpusid against paper_lookup.
# Expected: ~120 K overlapping papers between one paper shard and one abstract shard,
# so TARGET=1500 matches are found after scanning ~60 K abstract records (~160 s).
rows = []

for rec in tqdm(stream_gz_jsonl(abstract_files[SHARD_IDX], HEADERS), desc=f"scanning abstract shard {SHARD_IDX}"):
    corpusid = str(rec.get("corpusid") or "").strip()
    abstract = str(rec.get("abstract") or "").strip()
    if corpusid and abstract and corpusid in paper_lookup:
        rows.append({"corpusid": corpusid, "abstract": abstract, **paper_lookup[corpusid]})
    if len(rows) >= TARGET:
        break

print(f"Collected {len(rows)} joined papers")

scanning abstract shard 0: 40155it [00:01, 20999.59it/s]

Collected 1500 joined papers


In [29]:
# ---------- step 3: save ----------
joined_df = pd.DataFrame(rows)
joined_df["corpusid"] = joined_df["corpusid"].astype(str)
joined_df = normalize_and_save(joined_df, JOINED_OUTPUT_CSV, JOINED_OUTPUT_PARQUET)

Saved large_joined_sample.csv — shape: (1500, 10)
title coverage   : 100.00%
abstract coverage: 100.00%
    corpusid                                              title  \
0   15884564  C-Terminal Amino Acids 471-507 of Avian Hepati...   
1  251335420  Lethality of Police Shootings and Proximity to...   
2  238991506  Real world data on symptomology and diagnostic...   

                                            abstract    year  \
0  The infection of chickens with avian Hepatitis...  2016.0   
1  Studies of police shootings have typically foc...  2022.0   
2  Endometriosis is a chronic disease that requir...  2021.0   

                venue                                            authors  \
0            PLoS ONE  [{'authorId': '2107922154', 'name': 'Xinquan Z...   
1    Homicide Studies  [{'authorId': '118784192', 'name': 'Peter A. H...   
2  Scientific Reports  [{'authorId': '1632983093', 'name': 'K. Becker...   

                                     s2fieldsofstudy  citationcou